# Lab 10 — Part A: Lakehouse Federation

**Goal:** Query Neon PostgreSQL through a Unity Catalog foreign catalog and join external data with a local Delta table.

**External source:** Neon PostgreSQL

**Foreign catalog:** parameter `foreign_catalog` (Personal DEV default: `neon_postgres_conn_catalog`)

**Local Delta schema:** parameters `catalog`.`schema` (Personal DEV default: `workspace.lab_10_federation_cdc`)

This notebook demonstrates:
1. Querying external PostgreSQL data without copying it.
2. Joining external PostgreSQL orders with local Delta customers.
3. Understanding Federation versus Delta ingestion.

All operations are read-only.

In [ ]:
# WHAT: Read deployment parameters. Defaults are the Personal DEV values.
# WHY: The same notebook runs in Personal DEV and Azure PROD without edits;
#      the Azure Job passes its own catalog, schema and foreign catalog.
import re

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "lab_10_federation_cdc")
dbutils.widgets.text("foreign_catalog", "neon_postgres_conn_catalog")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
FOREIGN_CATALOG = dbutils.widgets.get("foreign_catalog").strip()

# Identifiers are interpolated into SQL, so only plain names are accepted.
for name, value in {"catalog": CATALOG, "schema": SCHEMA, "foreign_catalog": FOREIGN_CATALOG}.items():
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value), f"Invalid {name}: {value!r}"

BASE = f"{CATALOG}.{SCHEMA}"
print(f"Delta schema: {BASE}" + f" | foreign catalog: {FOREIGN_CATALOG}")

In [0]:
# WHAT: Read data directly from Neon PostgreSQL.
# WHY: Confirm that Lakehouse Federation is working.

customers = spark.sql(f"""
    SELECT COUNT(*) AS total
    FROM {FOREIGN_CATALOG}.public.lab10_customers
""").first()["total"]

orders = spark.sql(f"""
    SELECT COUNT(*) AS total
    FROM {FOREIGN_CATALOG}.public.lab10_orders
""").first()["total"]

print(f"Neon customers: {customers}")
print(f"Neon orders: {orders}")

assert customers == 12, "Unexpected customer count"
assert orders == 20, "Unexpected order count"

print("PASS: Lakehouse Federation")

In [0]:
# WHAT: Join external PostgreSQL orders with local Delta customers.
# WHY: Demonstrate a hybrid Lakehouse Federation query.

joined = spark.sql(f"""
    SELECT
        o.order_id,
        o.customer_id,
        c.first_name,
        c.city,
        o.amount,
        o.status,
        CASE
            WHEN c.customer_id IS NULL THEN 'NO_DELTA_MATCH'
            ELSE 'MATCHED'
        END AS join_status
    FROM {FOREIGN_CATALOG}.public.lab10_orders o
    LEFT JOIN {BASE}.customers_delta c
        ON o.customer_id = c.customer_id
    ORDER BY o.order_id
""")

display(joined)

# A LEFT JOIN must retain every external order.
assert joined.count() == orders, "Order count mismatch"

# Every order_id should appear exactly once.
assert joined.select("order_id").distinct().count() == orders

display(joined.groupBy("join_status").count())

print("PASS: Federation + Delta JOIN")

In [0]:
# WHAT: Inspect federated query planning.
# WHY: Understand how Databricks accesses external data.

federation_plan = spark.sql(f"""
    EXPLAIN EXTENDED
    SELECT customer_id, amount
    FROM {FOREIGN_CATALOG}.public.lab10_orders
    WHERE amount > 100
""")

display(federation_plan)

# Compare with a local Delta query.
delta_plan = spark.sql(f"""
    EXPLAIN EXTENDED
    SELECT customer_id, city
    FROM {BASE}.customers_delta
    WHERE city = 'Krakow'
""")

display(delta_plan)